In [3]:
import torch 
import torch.optim as optim
from utils import get_pennfudan_dataloaders
from models import ObjectCountingModel
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def train_model(num_epochs=10, batch_size=2, learning_rate=0.005):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

    train_loader, val_loader, category_names = get_pennfudan_dataloaders(batch_size)
    
    model = ObjectCountingModel(num_classes=2)
    model.to(device)

    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = optim.SGD(params, lr=learning_rate, momentum=0.3, weight_decay=0.0005)
    lr_scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

    map_metric = MeanAveragePrecision(iou_type='bbox').to(device)

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for images, targets in train_loader:
            images = [image.to(device) for image in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            
            running_loss += losses.item()
        
        lr_scheduler.step()
    
        model.eval()
        map_metric.reset()  # Reset for each epoch
        with torch.no_grad():
            for images, targets in val_loader:
                images = [image.to(device) for image in images]
                targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

                preds = model(images)
                map_metric.update(preds, targets)
        
        metrics = map_metric.compute()
        print(f'''Epoch {epoch + 1}/{num_epochs}, Loss: {running_loss / len(train_loader):.4f}
Val mAP: {metrics["map"]:.4f} (mAP@0.5: {metrics["map_50"]:.4f})''')

        # Save model
        torch.save(model.state_dict(), 'object_counting_model.pth')
    
    return model, category_names


In [4]:
train_model()

Epoch 1/10, Loss: 0.4523
Val mAP: 0.4682 (mAP@0.5: 0.8240)
Epoch 2/10, Loss: 0.3411
Val mAP: 0.5479 (mAP@0.5: 0.9220)


KeyboardInterrupt: 